In [1]:
pip install openai pandas openpyxl


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import re
import math
import json
import time
import pandas as pd
from getpass import getpass
from openai import OpenAI

# =========================================================
# 1) PATHS
# =========================================================
data_dir = "/Users/shinekhantaung/Desktop/GPT-NER/5fold_train_test"
output_dir = os.path.join(data_dir, "zero_shot_results")
os.makedirs(output_dir, exist_ok=True)

# =========================================================
# 2) API KEY (secure input)
# =========================================================
api_key = getpass("Enter your OpenAI API key: ")
client = OpenAI(api_key=api_key)

# =========================================================
# 3) SETTINGS
# =========================================================
MODEL_NAME = "gpt-4.1"

# column names: dataset နဲ့မကိုက်ရင် ပြင်ပါ
sentence_col = "sentence"
name_col = "gold_names"

PROMPT_TEMPLATE = """I am an excellent linguist.

The task is to label person entities in the given Old English sentence using @@ and ##.
If no person entity is present, return the sentence unchanged.

Sentence: {sentence}
Output:"""

# =========================================================
# 4) HELPERS
# =========================================================
def normalize_text(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s).strip()
    s = re.sub(r"\s+", " ", s)
    return s

def normalize_entity(s: str) -> str:
    s = normalize_text(s)
    return s.casefold()

def parse_gold_names(cell_value):
    """
    Gold name column က
    - blank / NaN
    - one name
    - multiple names separated by ; , |
    ဆိုတာတွေ handle လုပ်မယ်
    """
    if pd.isna(cell_value):
        return []

    text = str(cell_value).strip()
    if not text:
        return []

    parts = re.split(r"[;|,]", text)
    names = [normalize_text(x) for x in parts if normalize_text(x)]

    # deduplicate
    seen = set()
    result = []
    for n in names:
        key = normalize_entity(n)
        if key not in seen:
            seen.add(key)
            result.append(n)
    return result

def extract_marked_names(output_text: str):
    """
    @@name## pattern တွေ extract
    """
    if not isinstance(output_text, str):
        return []

    matches = re.findall(r"@@(.*?)##", output_text, flags=re.DOTALL)
    names = [normalize_text(m) for m in matches if normalize_text(m)]

    # deduplicate
    seen = set()
    result = []
    for n in names:
        key = normalize_entity(n)
        if key not in seen:
            seen.add(key)
            result.append(n)
    return result

def call_gpt(sentence: str):
    prompt = PROMPT_TEMPLATE.format(sentence=sentence)

    response = client.responses.create(
        model=MODEL_NAME,
        input=prompt,
    )

    # SDK versions အလိုက် output_text access ကွာနိုင်လို့ safe extraction
    output_text = getattr(response, "output_text", None)
    if output_text is None:
        try:
            output_text = response.output[0].content[0].text
        except Exception:
            output_text = str(response)

    return output_text.strip()

def compute_metrics(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    return precision, recall, f1

# =========================================================
# 5) RUN ALL 5 FOLDS
# =========================================================
fold_results = []
all_tp, all_fp, all_fn = 0, 0, 0

for fold_id in range(1, 6):
    test_path = os.path.join(data_dir, f"test_fold_{fold_id}.xlsx")
    if not os.path.exists(test_path):
        print(f"[Skip] File not found: {test_path}")
        continue

    df = pd.read_excel(test_path)

    # column name check
    if sentence_col not in df.columns or name_col not in df.columns:
        print(f"[Error] Fold {fold_id}: columns not found.")
        print("Available columns:", df.columns.tolist())
        continue

    tp, fp, fn = 0, 0, 0
    row_outputs = []

    print(f"\n========== Fold {fold_id} ==========")
    print(f"Rows: {len(df)}")

    for i, row in df.iterrows():
        sentence = normalize_text(row[sentence_col])
        gold_names = parse_gold_names(row[name_col])

        try:
            pred_output = call_gpt(sentence)
            pred_names = extract_marked_names(pred_output)
        except Exception as e:
            pred_output = f"[ERROR] {e}"
            pred_names = []

        gold_set = {normalize_entity(x) for x in gold_names}
        pred_set = {normalize_entity(x) for x in pred_names}

        row_tp = len(gold_set & pred_set)
        row_fp = len(pred_set - gold_set)
        row_fn = len(gold_set - pred_set)

        tp += row_tp
        fp += row_fp
        fn += row_fn

        row_outputs.append({
            "sentence": sentence,
            "gold_name_raw": row[name_col],
            "gold_names_parsed": "; ".join(gold_names),
            "model_output": pred_output,
            "pred_names_parsed": "; ".join(pred_names),
            "row_tp": row_tp,
            "row_fp": row_fp,
            "row_fn": row_fn,
        })

        if (i + 1) % 20 == 0:
            print(f"Processed {i + 1}/{len(df)}")

        time.sleep(0.2)  # rate limit soft protection

    precision, recall, f1 = compute_metrics(tp, fp, fn)

    print(f"TP={tp}, FP={fp}, FN={fn}")
    print(f"Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

    fold_results.append({
        "fold": fold_id,
        "rows": len(df),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    })

    all_tp += tp
    all_fp += fp
    all_fn += fn

    # save detailed predictions
    detail_df = pd.DataFrame(row_outputs)
    detail_out = os.path.join(output_dir, f"predictions_fold_{fold_id}.xlsx")
    detail_df.to_excel(detail_out, index=False)

# =========================================================
# 6) FINAL SUMMARY
# =========================================================
summary_df = pd.DataFrame(fold_results)

if not summary_df.empty:
    macro_precision = summary_df["precision"].mean()
    macro_recall = summary_df["recall"].mean()
    macro_f1 = summary_df["f1"].mean()

    micro_precision, micro_recall, micro_f1 = compute_metrics(all_tp, all_fp, all_fn)

    print("\n========== Fold-wise Results ==========")
    print(summary_df)

    print("\n========== Average Across 5 Folds ==========")
    print(f"Macro Precision: {macro_precision:.4f}")
    print(f"Macro Recall   : {macro_recall:.4f}")
    print(f"Macro F1       : {macro_f1:.4f}")

    print("\n========== Overall Micro Score ==========")
    print(f"TP={all_tp}, FP={all_fp}, FN={all_fn}")
    print(f"Micro Precision: {micro_precision:.4f}")
    print(f"Micro Recall   : {micro_recall:.4f}")
    print(f"Micro F1       : {micro_f1:.4f}")

    # save summary
    summary_out = os.path.join(output_dir, "fold_summary.xlsx")
    summary_df.to_excel(summary_out, index=False)

    final_txt = os.path.join(output_dir, "final_scores.txt")
    with open(final_txt, "w", encoding="utf-8") as f:
        f.write("Fold-wise Results\n")
        f.write(summary_df.to_string(index=False))
        f.write("\n\nAverage Across 5 Folds\n")
        f.write(f"Macro Precision: {macro_precision:.4f}\n")
        f.write(f"Macro Recall   : {macro_recall:.4f}\n")
        f.write(f"Macro F1       : {macro_f1:.4f}\n\n")
        f.write("Overall Micro Score\n")
        f.write(f"TP={all_tp}, FP={all_fp}, FN={all_fn}\n")
        f.write(f"Micro Precision: {micro_precision:.4f}\n")
        f.write(f"Micro Recall   : {micro_recall:.4f}\n")
        f.write(f"Micro F1       : {micro_f1:.4f}\n")

    print(f"\nSaved results to: {output_dir}")
else:
    print("No fold results were generated.")


========== Fold 1 ==========
Rows: 186
Processed 20/186
Processed 40/186
Processed 60/186
Processed 80/186
Processed 100/186
Processed 120/186
Processed 140/186
Processed 160/186
Processed 180/186
TP=225, FP=61, FN=47
Precision=0.7867, Recall=0.8272, F1=0.8065

========== Fold 2 ==========
Rows: 186
Processed 20/186
Processed 40/186
Processed 60/186
Processed 80/186
Processed 100/186
Processed 120/186
Processed 140/186
Processed 160/186
Processed 180/186
TP=211, FP=59, FN=37
Precision=0.7815, Recall=0.8508, F1=0.8147

========== Fold 3 ==========
Rows: 186
Processed 20/186
Processed 40/186
Processed 60/186
Processed 80/186
Processed 100/186
Processed 120/186
Processed 140/186
Processed 160/186
Processed 180/186
TP=200, FP=59, FN=31
Precision=0.7722, Recall=0.8658, F1=0.8163

========== Fold 4 ==========
Rows: 186
Processed 20/186
Processed 40/186
Processed 60/186
Processed 80/186
Processed 100/186
Processed 120/186
Processed 140/186
Processed 160/186
Processed 180/186
TP=209, FP=62, F